In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df = pd.read_csv("clusters_candidates.tsv", sep="\t")
df.head()

,MAG,query,target,identity,aln_len,evalue,bitscore,product,contig,start,...,Organism name,Locus,Gene,Function,PFAM ID,Reference,mid,distance,cluster,cluster_size
0,GCA_902754875.1,EJF39103.1,GCA_902754875.1_KMPMIM_00214,34.9,364,5.160000e-64,212.0,hypothetical protein,CACVLA010000002.1,26588,...,Clostridium sp. MSTE9,sulfo-TK,sqwF,SA reductase,PF00465,https://zhaogroup.chbe.illinois.edu/publicatio...,27136.5,NaN,14,4
1,GCA_902754875.1,EJF39089.1,GCA_902754875.1_KMPMIM_00215,57.6,85,4.140000e-28,103.0,hypothetical protein,CACVLA010000002.1,27675,...,Clostridium sp. MSTE9,sulfo-TK,sqwI,SE isomerase,PF02502,https://zhaogroup.chbe.illinois.edu/publicatio...,27803.5,667.0,14,4
2,GCA_902754875.1,EJF39090.1,GCA_902754875.1_KMPMIM_00216,55.1,254,7.350000e-88,267.0,hypothetical protein,CACVLA010000002.1,27946,...,Clostridium sp. MSTE9,sulfo-TK,sqwH,SF transketolase,"PF02779, PF02780",https://zhaogroup.chbe.illinois.edu/publicatio...,28326.5,523.0,14,4
3,GCA_902754875.1,AYG21321.1,GCA_902754875.1_KMPMIM_00222,24.5,277,1.160000e-05,50.4,hypothetical protein,CACVLA010000002.1,32567,...,Escherichia coli str. K-12 substr. MG1655,sulfo-EMP,yihV,SF kinase,PF00294,https://www.nature.com/articles/nature12947,32961.0,4634.5,14,4
4,GCA_902754875.1,EJF39099.1,GCA_902754875.1_KMPMIM_01959,50.6,265,7.450000e-85,259.0,Transketolase 1,CACVLA010000043.1,5168,...,Clostridium sp. MSTE9,sulfo-TK,sqwG,SF transketolase,PF00456,https://zhaogroup.chbe.illinois.edu/publicatio...,5590.5,NaN,27,2


In [4]:
# Всего точек до фильтрации
total_before = len(df.index)

# Применённый фильтр
df_q = df[
    (df['evalue'] < 1e-5) &
    (df['bitscore'] > 50)
].copy()

# Всего точек после фильтрации
total_after = len(df_q)

# Сколько отброшено
dropped = total_before - total_after
dropped_percent = dropped / total_before * 100

print(f"Всего точек до фильтрации: {total_before}")
print(f"Всего точек после фильтрации: {total_after}")
print(f"Отброшено: {dropped} ({dropped_percent:.1f}%)")

Всего точек до фильтрации: 9740
Всего точек после фильтрации: 9232
Отброшено: 508 (5.2%)


In [5]:
# сначала собираем локусы
loci = df_q.groupby('cluster').agg({
    'Gene': lambda x: ','.join(sorted(set(x.dropna())))
}).reset_index()

In [6]:
loci.head()

,cluster,Gene
0,14,"sqwF,sqwH,sqwI"
1,27,"sqwG,sqwH"
2,35,"sqvD,sqwF,sqwG,sqwH"
3,38,"sqwG,sqwI"
4,56,"sqwG,sqwH"


In [ ]:
def annotate_sulfo_pathways(gene_list):
    genes = {g.lower() for g in gene_list}

    results = {}

    # universal upstream
    #upstream = genes & {"yihq", "sqga", "yihs", "sqvd"}

    up_block1 = bool(genes & {"yihq", "sqga"})
    up_block2 = bool(genes & {"yihs", "sqvd"})

    n_up_blocks = sum([up_block1, up_block2])

    # pathway-specific downstream
    downstream_map = {
        "sulfo-EMP": {"yihu", "slab"},
        "sulfo-ED": {"yihu", "slab"},
        "sulfo-TAL": {"yihu", "slab"},
        "sulfo-TK": {"sqwk", "sqwl"},
        "sulfo-ASMO": {"squf"},
        "sulfo-ASDO": {"squf"},
    }

    # __CORE DETECTION__
    core_hits = {}

    # Sulfo-EMP
    emp_block1 = genes & {"sqia", "yiht"}
    emp_block2 = genes & {"sqik", "yihv"}
    if bool(emp_block1) and bool(emp_block2):
        core_hits["sulfo-EMP"] = True

    # Sulfo-ED
    ed_core = {"seda", "sedb", "sedc", "sedd"}
    if len(genes & ed_core) >= 3:
        core_hits["sulfo-ED"] = True

    # Sulfo-TK
    tk_block1 = {"sqwg", "sqwh"} <= genes
    tk_block2 = "sqwi" in genes
    tk_block3 = bool(genes & {"sqwf", "sqwd"})
    if sum([tk_block1, tk_block2, tk_block3]) >= 3:
        core_hits["sulfo-TK"] = True

    # Sulfo-TAL
    if "sqva" in genes:
        core_hits["sulfo-TAL"] = True

    # Sulfo-ASMO
    if "squd" in genes:
        core_hits["sulfo-ASMO"] = True

    # Sulfo-ASDO
    if "sqod" in genes:
        core_hits["sulfo-ASDO"] = True

    
    # __SCORING__
    for pathway in core_hits:

        has_up = n_up_blocks >= 1
        has_down = bool(genes & downstream_map[pathway])

        if has_up and has_down:
            score = "max" #помимо кора есть 1+ upstream И 1+ downstream
        elif has_up:
            score = "min_up" #помимо необходимого кора есть хотя бы один upstream
        elif has_down:
            score = "min_down" #помимо необходимого кора есть хотя бы один downstream
        else:
            score = "min" #есть только достаточное количество коровых белков

        results[pathway] = {
            "score": score,
            "n_up": n_up_blocks,
            "genes": genes
    }

    return results

In [13]:
def annotate_sulfo_pathways(gene_list):
    genes = {g.lower() for g in gene_list}

    results = {}

    # --- upstream ---
    up_block1 = bool(genes & {"yihq", "sqga"})
    up_block2 = bool(genes & {"yihs", "sqvd"})
    n_up_blocks = sum([up_block1, up_block2])

    # --- downstream ---
    downstream_map = {
        "sulfo-EMP": {"yihu", "slab"},
        "sulfo-ED": {"yihu", "slab"},
        "sulfo-TAL": {"yihu", "slab"},
        "sulfo-TK": {"sqwk", "sqwl"},
        "sulfo-ASMO": {"squf"},
        "sulfo-ASDO": {"squf"},
    }

    # --- CORE SCORING ---
    core_info = {}

    # EMP
    emp_hits = len(genes & {"sqia", "yiht", "sqik", "yihv"})
    core_info["sulfo-EMP"] = {
        "score": emp_hits,
        "is_core": emp_hits >= 2,
        "is_strict": (
            bool(genes & {"sqia", "yiht"}) and
            bool(genes & {"sqik", "yihv"})
        )
    }

    # ED
    ed_core = {"seda", "sedb", "sedc", "sedd"}
    ed_hits = len(genes & ed_core)
    core_info["sulfo-ED"] = {
        "score": ed_hits,
        "is_core": ed_hits >= 3,
        "is_strict": ed_hits >= 3
    }

    # TK
    tk_hits = sum([
        {"sqwg", "sqwh"} <= genes,
        "sqwi" in genes,
        bool(genes & {"sqwf", "sqwd"})
    ])
    core_info["sulfo-TK"] = {
        "score": tk_hits,
        "is_core": tk_hits >= 3,
        "is_strict": tk_hits >= 3
    }

    # TAL
    core_info["sulfo-TAL"] = {
        "score": int("sqva" in genes),
        "is_core": "sqva" in genes,
        "is_strict": "sqva" in genes
    }

    # ASMO
    core_info["sulfo-ASMO"] = {
        "score": int("squd" in genes),
        "is_core": "squd" in genes,
        "is_strict": "squd" in genes
    }

    # ASDO
    core_info["sulfo-ASDO"] = {
        "score": int("sqod" in genes),
        "is_core": "sqod" in genes,
        "is_strict": "sqod" in genes
    }

    # --- FILTER: только реальные core ---
    core_hits = {
        p: v for p, v in core_info.items() if v["is_core"]
    }

    # --- SCORING ---
    for pathway, info in core_hits.items():

        has_up = n_up_blocks >= 1
        has_down = bool(genes & downstream_map[pathway])

        if has_up and has_down:
            score = "max"
        elif has_up:
            score = "min_up"
        elif has_down:
            score = "min_down"
        else:
            score = "min"

        results[pathway] = {
            "score": score,
            "core_score": info["score"],
            "n_up": n_up_blocks
        }

    # --- ГИБРИДНОСТЬ ---
    n_core_paths = len(core_hits)

    is_hybrid = n_core_paths >= 2

    if n_core_paths == 0:
        locus_type = "none"
        label = None
    elif n_core_paths == 1:
        locus_type = "single"
        label = list(core_hits.keys())[0]
    else:
        locus_type = "hybrid"
        label = "+".join(sorted(core_hits.keys()))

    return {
        "pathways": results,
        "n_core_paths": n_core_paths,
        "is_hybrid": is_hybrid,
        "type": locus_type,
        "label": label
    }

In [15]:
# apply to df
loci["gene_set"] = loci["Gene"].str.split(',').apply(set)

loci["annotation"] = loci["gene_set"].apply(annotate_sulfo_pathways)

In [20]:
loci[loci["annotation"].apply(lambda x: x["n_core_paths"] >= 1)]

,cluster,Gene,gene_set,annotation
5,63,"sqiA,sqiK","{sqiK, sqiA}","{'pathways': {'sulfo-EMP': {'score': 'min', 'c..."
6,66,"sqvB,sqvD,sqwF,sqwG,sqwH,sqwI","{sqvB, sqwF, sqwI, sqwG, sqwH, sqvD}","{'pathways': {'sulfo-TK': {'score': 'min_up', ..."
22,271,"sqvA,sqwG,sqwH","{sqwH, sqwG, sqvA}","{'pathways': {'sulfo-TAL': {'score': 'min', 'c..."
35,387,"sqiA,sqiK","{sqiK, sqiA}","{'pathways': {'sulfo-EMP': {'score': 'min', 'c..."
36,392,"sqvD,sqwF,sqwG,sqwH,sqwI,yihQ","{yihQ, sqwF, sqwI, sqwG, sqwH, sqvD}","{'pathways': {'sulfo-TK': {'score': 'min_up', ..."
...,...,...,...,...
4217,37967,"sqiA,sqvA,sqwG","{sqiA, sqwG, sqvA}","{'pathways': {'sulfo-TAL': {'score': 'min', 'c..."
4221,38001,"sqvA,sqvB,sqvD,yihQ,yihU","{yihQ, sqvB, sqvA, yihU, sqvD}","{'pathways': {'sulfo-TAL': {'score': 'max', 'c..."
4239,38167,"sqvA,sqvB,sqvD","{sqvD, sqvB, sqvA}","{'pathways': {'sulfo-TAL': {'score': 'min_up',..."
4242,38187,"sqiA,sqiK","{sqiK, sqiA}","{'pathways': {'sulfo-EMP': {'score': 'min', 'c..."


In [21]:
loci.head(10)

,cluster,Gene,gene_set,annotation
0,14,"sqwF,sqwH,sqwI","{sqwF, sqwI, sqwH}","{'pathways': {}, 'n_core_paths': 0, 'is_hybrid..."
1,27,"sqwG,sqwH","{sqwH, sqwG}","{'pathways': {}, 'n_core_paths': 0, 'is_hybrid..."
2,35,"sqvD,sqwF,sqwG,sqwH","{sqwF, sqwH, sqwG, sqvD}","{'pathways': {}, 'n_core_paths': 0, 'is_hybrid..."
3,38,"sqwG,sqwI","{sqwI, sqwG}","{'pathways': {}, 'n_core_paths': 0, 'is_hybrid..."
4,56,"sqwG,sqwH","{sqwH, sqwG}","{'pathways': {}, 'n_core_paths': 0, 'is_hybrid..."
5,63,"sqiA,sqiK","{sqiK, sqiA}","{'pathways': {'sulfo-EMP': {'score': 'min', 'c..."
6,66,"sqvB,sqvD,sqwF,sqwG,sqwH,sqwI","{sqvB, sqwF, sqwI, sqwG, sqwH, sqvD}","{'pathways': {'sulfo-TK': {'score': 'min_up', ..."
7,104,"sedA,sqwI","{sedA, sqwI}","{'pathways': {}, 'n_core_paths': 0, 'is_hybrid..."
8,120,"sqwG,sqwH","{sqwH, sqwG}","{'pathways': {}, 'n_core_paths': 0, 'is_hybrid..."
9,135,"sqiK,sqwI","{sqiK, sqwI}","{'pathways': {}, 'n_core_paths': 0, 'is_hybrid..."


In [35]:
loci[loci["annotation"].apply(lambda x: "sulfo-TAL" in x["pathways"])]

,cluster,Gene,gene_set,annotation,pathway_set
22,271,"sqvA,sqwG,sqwH","{sqwH, sqwG, sqvA}","{'pathways': {'sulfo-TAL': {'score': 'min', 'c...",{sulfo-TAL}
46,489,"sqvA,sqwG","{sqwG, sqvA}","{'pathways': {'sulfo-TAL': {'score': 'min', 'c...",{sulfo-TAL}
81,741,"sqvA,sqwG,sqwH","{sqwH, sqwG, sqvA}","{'pathways': {'sulfo-TAL': {'score': 'min', 'c...",{sulfo-TAL}
84,761,"sedA,sqvA","{sedA, sqvA}","{'pathways': {'sulfo-TAL': {'score': 'min', 'c...",{sulfo-TAL}
125,1068,"sqvA,sqwG,sqwH","{sqwH, sqwG, sqvA}","{'pathways': {'sulfo-TAL': {'score': 'min', 'c...",{sulfo-TAL}
...,...,...,...,...,...
4177,37691,"sqvA,sqwG","{sqwG, sqvA}","{'pathways': {'sulfo-TAL': {'score': 'min', 'c...",{sulfo-TAL}
4217,37967,"sqiA,sqvA,sqwG","{sqiA, sqwG, sqvA}","{'pathways': {'sulfo-TAL': {'score': 'min', 'c...",{sulfo-TAL}
4221,38001,"sqvA,sqvB,sqvD,yihQ,yihU","{yihQ, sqvB, sqvA, yihU, sqvD}","{'pathways': {'sulfo-TAL': {'score': 'max', 'c...",{sulfo-TAL}
4239,38167,"sqvA,sqvB,sqvD","{sqvD, sqvB, sqvA}","{'pathways': {'sulfo-TAL': {'score': 'min_up',...",{sulfo-TAL}


In [32]:
loci["pathway_set"] = loci["annotation"].apply(lambda x: set(x["pathways"].keys()))

In [33]:
loci[loci["pathway_set"].apply(lambda s: "sulfo-TAL" in s)]

,cluster,Gene,gene_set,annotation,pathway_set
22,271,"sqvA,sqwG,sqwH","{sqwH, sqwG, sqvA}","{'pathways': {'sulfo-TAL': {'score': 'min', 'c...",{sulfo-TAL}
46,489,"sqvA,sqwG","{sqwG, sqvA}","{'pathways': {'sulfo-TAL': {'score': 'min', 'c...",{sulfo-TAL}
81,741,"sqvA,sqwG,sqwH","{sqwH, sqwG, sqvA}","{'pathways': {'sulfo-TAL': {'score': 'min', 'c...",{sulfo-TAL}
84,761,"sedA,sqvA","{sedA, sqvA}","{'pathways': {'sulfo-TAL': {'score': 'min', 'c...",{sulfo-TAL}
125,1068,"sqvA,sqwG,sqwH","{sqwH, sqwG, sqvA}","{'pathways': {'sulfo-TAL': {'score': 'min', 'c...",{sulfo-TAL}
...,...,...,...,...,...
4177,37691,"sqvA,sqwG","{sqwG, sqvA}","{'pathways': {'sulfo-TAL': {'score': 'min', 'c...",{sulfo-TAL}
4217,37967,"sqiA,sqvA,sqwG","{sqiA, sqwG, sqvA}","{'pathways': {'sulfo-TAL': {'score': 'min', 'c...",{sulfo-TAL}
4221,38001,"sqvA,sqvB,sqvD,yihQ,yihU","{yihQ, sqvB, sqvA, yihU, sqvD}","{'pathways': {'sulfo-TAL': {'score': 'max', 'c...",{sulfo-TAL}
4239,38167,"sqvA,sqvB,sqvD","{sqvD, sqvB, sqvA}","{'pathways': {'sulfo-TAL': {'score': 'min_up',...",{sulfo-TAL}


In [34]:
from collections import Counter

Counter(
    loci["pathway_set"].apply(lambda x: "+".join(sorted(x)))
)

Counter({'': 3858,
         'sulfo-TAL': 175,
         'sulfo-EMP': 166,
         'sulfo-TK': 50,
         'sulfo-ASMO': 5,
         'sulfo-TAL+sulfo-TK': 2,
         'sulfo-ED': 1})

In [36]:
# annotation expantion
expanded = pd.concat([
    pd.DataFrame([
        {
            "cluster": row.cluster,
            "SQ_pathway": p,
            "SQ_score": v["score"],
            "SQ_gene_content_set": ",".join(sorted(row.gene_set)),
            "SQ_label": row.annotation["label"],          # ← НОВОЕ (очень полезно)
            "SQ_type": row.annotation["type"]             # ← НОВОЕ (single/hybrid)
        }
        for p, v in row.annotation["pathways"].items()
    ])
    for _, row in loci[
        loci["annotation"].apply(lambda x: len(x["pathways"]) > 0)
    ].iterrows()
], ignore_index=True)

# clusters metadata
meta = df.groupby("cluster").agg(
    contig=("contig", "first"),
    taxonomy=("taxonomy", "first"),
    target=("target", lambda x: ",".join(sorted(set(x)))),
    bitscore_list=("bitscore", lambda x: list(x)),
    evalue_list=("evalue", lambda x: list(x))
).reset_index()

# merge
final_table = expanded.merge(meta, on="cluster", how="left")

final_table = final_table[
    ["cluster", "target", "contig", "SQ_gene_content_set",
     "SQ_pathway", "SQ_score", "taxonomy", "bitscore_list", "evalue_list"]
]

In [37]:
final_table.head()

,cluster,target,contig,SQ_gene_content_set,SQ_pathway,SQ_score,taxonomy,bitscore_list,evalue_list
0,63,"GCA_902754915.1_GLIGFB_00196,GCA_902754915.1_G...",CACVLH010000001.1,"sqiA,sqiK",sulfo-EMP,min,d__Bacteria;p__Bacillota;c__Clostridia;o__Osci...,"[182.0, 185.0]","[7.299999999999999e-52, 4.5899999999999996e-55]"
1,66,"GCA_902754915.1_GLIGFB_00381,GCA_902754915.1_G...",CACVLH010000002.1,"sqvB,sqvD,sqwF,sqwG,sqwH,sqwI",sulfo-TK,min_up,d__Bacteria;p__Bacillota;c__Clostridia;o__Osci...,"[142.0, 217.0, 162.0, 291.0, 298.0, 255.0]","[1.6599999999999999e-40, 5.99e-66, 3.31e-50, 1..."
2,271,"GCA_902755405.1_GNFANA_00528,GCA_902755405.1_G...",CACVMX010000032.1,"sqvA,sqwG,sqwH",sulfo-TAL,min,d__Bacteria;p__Bacillota;c__Clostridia;o__Osci...,"[232.0, 227.0, 148.0]","[2.86e-74, 1.79e-71, 3.93e-43]"
3,387,"GCA_902755455.1_HAJFOH_00002,GCA_902755455.1_H...",CACVNO010000001.1,"sqiA,sqiK",sulfo-EMP,min,d__Bacteria;p__Bacillota;c__Clostridia;o__Osci...,"[160.0, 182.0]","[1.03e-45, 7.299999999999999e-52]"
4,392,"GCA_902755455.1_HAJFOH_00507,GCA_902755455.1_H...",CACVNO010000004.1,"sqvD,sqwF,sqwG,sqwH,sqwI,yihQ",sulfo-TK,min_up,d__Bacteria;p__Bacillota;c__Clostridia;o__Osci...,"[391.0, 275.0, 295.0, 292.0, 159.0, 206.0]","[1.12e-125, 1.51e-86, 3.47e-99, 8.999999999999..."


In [38]:
final_table.to_csv("final_table.csv", index=False)